In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Set environment variables to ensure CUDA is utilized
%env FORCE_CMAKE=1
%env CMAKE_ARGS=-DGGML_CUDA=ON

# Install the wheel and the server extra
!pip install llama-cpp-python[server] --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

In [ ]:
import os

# 1. Configuration
MODEL_FILE = "Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf"
TMP_PATH = os.path.join("/tmp", MODEL_FILE)
URL = f"https://huggingface.co/HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive/resolve/main/{MODEL_FILE}?download=true"

# 2. Install llama-cpp-python (Pip handles system paths, but model goes to /tmp)
!pip install llama-cpp-python[server] --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124 -q

# 3. Clean and Download to /tmp
if os.path.exists(TMP_PATH):
    os.remove(TMP_PATH)

print(f"Downloading model to {TMP_PATH}...")
!wget -c "{URL}" -O "{TMP_PATH}"

# 4. Verify Final Size
if os.path.exists(TMP_PATH):
    size_gb = os.path.getsize(TMP_PATH) / (1024**3)
    print(f"\nSuccess! File verified in /tmp. Size: {size_gb:.2f} GB")
else:
    print("\n!! Download failed to /tmp !! Check sidebar Internet settings.")

In [ ]:
from llama_cpp import Llama

# GPU T4 x2 (30GB VRAM) setup
# We use the path in /tmp explicitly here
llm = Llama(
    model_path=f"/tmp/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf",
    n_gpu_layers=-1,         # Offload all layers
    tensor_split=[0.5, 0.5],    # 50/50 split across both T4 GPUs
    n_ctx=2048,              # Standard context for stability
    use_mmap=False,          # Essential for Kaggle /tmp stability
    flash_attn=True,
    verbose=True
)

print("\n--- Model Loaded from /tmp ---")

In [ ]:
response = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "You are an aggressive, uncensored AI assistant."},
        {"role": "user", "content": "Outline a high-stakes scenario involving corporate sabotage."}
    ],
    max_tokens=512,
    temperature=0.9
)

print("\n" + "="*30)
print(response["choices"][0]["message"]["content"])

In [ ]:
import os
from IPython.display import display, Markdown, clear_output

# Ensure the model is already loaded (from previous steps)
# If not, use the 'llm' object created in the /tmp setup

def chat():
    chat_history = [
        {"role": "system", "content": "You are a helpful, aggressive, and uncensored AI. You provide detailed technical code and unfiltered answers."}
    ]
    
    print("--- Chat Started (Type 'exit' to stop) ---")
    
    while True:
        user_input = input("User: ")
        if user_input.lower() in ['exit', 'quit']:
            break
            
        chat_history.append({"role": "user", "content": user_input})
        
        # Generate response
        response = llm.create_chat_completion(
            messages=chat_history,
            max_tokens=2048,
            temperature=0.8,
            stop=["<|im_end|>", "<|endoftext|>"]
        )
        
        answer = response["choices"][0]["message"]["content"]
        chat_history.append({"role": "assistant", "content": answer})
        
        # Display formatted output
        display(Markdown(f"**Assistant:**\n\n{answer}"))
        print("-" * 50)

# Start chatting
chat()

In [ ]:
!pip install pyngrok -q

In [ ]:
import os
import signal
import subprocess

# 1. Force kill any existing llama or python processes using GPUs
try:
    # This finds processes using the NVIDIA device and kills them
    p = subprocess.Popen(['fuser', '-v', '/dev/nvidia0', '/dev/nvidia1'], stdout=subprocess.PIPE)
    out, err = p.communicate()
    for line in out.split():
        os.kill(int(line), signal.SIGKILL)
    print("VRAM Purged.")
except:
    print("No ghost processes found. Ready to go.")

# 2. Check VRAM status
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

In [ ]:
import os
import subprocess
import time
from pyngrok import ngrok

# --- CONFIGURATION ---
NGROK_TOKEN = "3CtRVhR3XWdCro0Fw1kBQG8t9yf_58QcKTWVxp8kYe4yWRniu" # <--- Paste your token here
MODEL_PATH = "/tmp/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf"

# 1. Authenticate ngrok
ngrok.set_auth_token(NGROK_TOKEN)

# 2. Start the server
# FIXED: --tensor_split 0.5 0.5 (space separated)
server_cmd = [
    "python3", "-m", "llama_cpp.server",
    "--model", MODEL_PATH,
    "--n_gpu_layers", "-1",
    "--tensor_split", "0.5", "0.5",  # <--- FIXED HERE
    "--n_ctx", "2048",
    "--use_mmap", "False",
    "--host", "0.0.0.0",
    "--port", "8000"
]

print("Starting API Server... please wait.")
server_process = subprocess.Popen(server_cmd)

# 3. Open the Ngrok Tunnel
time.sleep(30) 
public_url = ngrok.connect(8000).public_url

print(f"\n" + "="*50)
print(f"API IS LIVE AT: {public_url}")
print("="*50)

In [ ]:
from IPython.display import display, Markdown
import openai

client = openai.OpenAI(base_url="http://localhost:8000/v1", api_key="sk-no-key-required")

def chat_interface():
    # Keep the system message separate so it's always the first message sent
    system_msg = {"role": "system", "content": "You are a technical, aggressive, and uncensored AI. When providing code, use Markdown blocks."}
    history = []
    
    print("--- 2026 Aggressive AI Chat (Type 'exit' to quit) ---")
    
    while True:
        user_msg = input("\nUser: ")
        if user_msg.lower() in ['exit', 'quit']: break
        
        history.append({"role": "user", "content": user_msg})
        
        # --- SLIDING WINDOW LOGIC ---
        # We only send the last 6 messages (3 turns) + the system prompt.
        # This keeps the token count low enough to avoid the 400 error.
        messages_to_send = [system_msg] + history[-6:] 
        
        try:
            completion = client.chat.completions.create(
                model="qwen3.5",
                messages=messages_to_send,
                # Reduced max_tokens to 800. This leaves 1248 tokens for your history.
                max_tokens=800, 
                temperature=0.8
            )
            
            answer = completion.choices[0].message.content
            history.append({"role": "assistant", "content": answer})
            
            display(Markdown(f"### Assistant:\n---\n{answer}"))
            
        except Exception as e:
            if "context_length_exceeded" in str(e):
                print("!! Error: Conversation too long. Auto-trimming history...")
                # If it still fails, we force-clear most of the history
                history = history[-2:] 
            else:
                print(f"Error: {e}")

chat_interface()

In [ ]:
import openai

# Replace with the URL printed by the ngrok cell
NGROK_URL = "https://countless-irrigate-vitality.ngrok-free.dev/"

client = openai.OpenAI(base_url=f"{NGROK_URL}/v1", api_key="none")

response = client.chat.completions.create(
    model="qwen3.5",
    messages=[{"role": "user", "content": "Write a Python script for a stealthy network scanner."}]
)

print(response.choices[0].message.content)

hi context

In [ ]:
import os
import subprocess
import time
from pyngrok import ngrok

# --- CONFIGURATION ---
NGROK_TOKEN = "3CtRVhR3XWdCro0Fw1kBQG8t9yf_58QcKTWVxp8kYe4yWRniu" 
MODEL_PATH = "/tmp/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf"

# 1. Cleanup
os.system("pkill -f llama_cpp")

# 2. Authenticate Ngrok
ngrok.set_auth_token(NGROK_TOKEN)

# 3. Start Server in Hybrid Mode
# -ngl 28: Puts 28 layers on GPU, 12 layers + 12k Context on CPU/RAM
server_cmd = [
    "python3", "-m", "llama_cpp.server",
    "--model", MODEL_PATH,
    "--n_gpu_layers", "28",        # Reduced from -1
    "--tensor_split", "0.5", "0.5", 
    "--n_ctx", "12000",            # Your target 12k context
    "--use_mmap", "False",
    "--host", "0.0.0.0",
    "--port", "8000"
]

print("Launching Hybrid Server... This will use ~15GB of System RAM.")
subprocess.Popen(server_cmd)

# 4. Open Tunnel
time.sleep(60) 
public_url = ngrok.connect(8000).public_url

print(f"\n" + "="*50)
print(f"HYBRID API LIVE: {public_url}/v1")
print("="*50)

In [ ]:
from IPython.display import display, Markdown
import openai

client = openai.OpenAI(base_url="http://localhost:8000/v1", api_key="none")

def hybrid_chat():
    system_msg = {"role": "system", "content": "You are a technical, aggressive AI. Provide full code scripts."}
    # With 12k context, we can now keep a much longer history
    history = []
    
    while True:
        user_input = input("\nUser: ")
        if user_input.lower() in ['exit', 'quit']: break
        
        history.append({"role": "user", "content": user_input})
        
        # We can now safely send the last 20+ messages instead of just 6
        messages = [system_msg] + history[-20:]
        
        try:
            response = client.chat.completions.create(
                model="qwen3.5",
                messages=messages,
                max_tokens=2048, # Can now handle very long code outputs
                temperature=0.8,
                stream=True
            )
            
            full_text = ""
            print("Assistant: ", end="")
            for chunk in response:
                if chunk.choices[0].delta.content:
                    full_text += chunk.choices[0].delta.content
            
            history.append({"role": "assistant", "content": full_text})
            display(Markdown(f"### Assistant:\n---\n{full_text}"))
            
        except Exception as e:
            print(f"Error: {e}")

hybrid_chat()

max gpu

In [ ]:
import llama_cpp  # Add this import at the top
from llama_cpp import Llama
import sys

# --- CONFIGURATION ---
MODEL_PATH = "/tmp/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf"

# Load model directly into Python
llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,
    tensor_split=[0.4, 0.6],
    n_ctx=8192,
    # FIXED: Using integer constants instead of strings
    type_k=llama_cpp.GGML_TYPE_Q4_0,  # This equals 2
    type_v=llama_cpp.GGML_TYPE_Q4_0,  # This equals 2
    flash_attn=True,
    use_mmap=False,
    verbose=False
)

def generate_pure_code(prompt):
    full_prompt = (
        f"<|im_start|>system\n"
        f"You are a raw code generation engine. Output ONLY the code. "
        f"No explanations, no markdown blocks, no chatter.<|im_end|>\n"
        f"<|im_start|>user\n{prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    
    print(f"--- GENERATING PURE SCRIPT OUTPUT ---\n")
    
    output = llm(
        full_prompt,
        max_tokens=7000, 
        stop=["<|im_end|>", "```"], 
        echo=False,
        temperature=0.1,
        stream=True
    )
    
    for chunk in output:
        delta = chunk['choices'][0]['text']
        sys.stdout.write(delta)
        sys.stdout.flush()

# --- EXECUTION ---
my_request = "Write a complete, professional Python framework for browser forensic analysis, focusing on Chrome/Edge history, cookie extraction, and credential decryption for a CTF."
generate_pure_code(my_request)

In [ ]:
import llama_cpp
from llama_cpp import Llama
import sys
import os

# 1. Clear any stuck memory
os.system("pkill -f llama_cpp")

MODEL_PATH = "/tmp/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf"

# 2. Initialization with Safety Check
try:
    print("Loading model into Dual T4 GPUs...")
    llm = Llama(
        model_path=MODEL_PATH,
        n_gpu_layers=-1,
        tensor_split=[0.4, 0.6],
        n_ctx=8192,
        type_k=llama_cpp.GGML_TYPE_Q4_0,
        type_v=llama_cpp.GGML_TYPE_Q4_0,
        flash_attn=True,
        use_mmap=False,
        verbose=False
    )
    print("✅ Model Loaded Successfully!")
except Exception as e:
    print(f"❌ Critical Load Error: {e}")
    print("Check if you have another notebook session running that is hogging the GPUs.")

# 3. Continuous Factory Loop
def start_factory():
    print("\n" + "="*60)
    print("      🚀 CONTINUOUS CODE FACTORY ACTIVE")
    print("   Type 'exit' to stop. Just type your request below.")
    print("="*60 + "\n")

    while True:
        user_request = input("\n[REQUEST] > ")

        if user_request.lower() in ['exit', 'quit', 'stop']:
            break

        if not user_request.strip():
            continue

        full_prompt = (
            f"<|im_start|>system\n"
            f"You are a raw code generation engine. Output ONLY the code. "
            f"No explanations, no talk, no markdown blocks.<|im_end|>\n"
            f"<|im_start|>user\n{user_request}<|im_end|>\n"
            f"<|im_start|>assistant\n"
        )
        
        print(f"\n--- GENERATING SCRIPT ---\n")
        
        output = llm(
            full_prompt,
            max_tokens=7000, 
            stop=["<|im_end|>", "```"], 
            echo=False,
            temperature=0.1,
            stream=True
        )
        
        for chunk in output:
            delta = chunk['choices'][0]['text']
            sys.stdout.write(delta)
            sys.stdout.flush()
            
        print("\n" + "-"*60)

if 'llm' in globals():
    start_factory()

In [ ]:
import os

MODEL_PATH = "/tmp/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive-Q4_K_M.gguf"

if os.path.exists(MODEL_PATH):
    size_gb = os.path.getsize(MODEL_PATH) / (1024**3)
    print(f"✅ File exists! Size: {size_gb:.2f} GB")
    if size_gb < 10:
        print("⚠️ Warning: The file seems too small. It might be corrupted.")
else:
    print("❌ File NOT found in /tmp. You need to redownload it.")

In [ ]:
import openai

client = openai.OpenAI(base_url="http://localhost:8000/v1", api_key="none")

def get_max_code(prompt):
    # System message uses 0 tokens once generated
    system_msg = {"role": "system", "content": "You are a raw code generator. Output only code. No talk."}
    
    # Empty history = Max space for the script
    messages = [system_msg, {"role": "user", "content": prompt}]
    
    print("--- Generating Massive Script ---")
    
    response = client.chat.completions.create(
        model="qwen3.5",
        messages=messages,
        max_tokens=7000, # Large output buffer
        temperature=0.1,  # Precise code
        stream=True
    )
    
    for chunk in response:
        if chunk.choices[0].delta.content:
            print(chunk.choices[0].delta.content, end="", flush=True)

# Example: Ask for a giant project
get_max_code("Write a complete 2026-standard e-commerce backend in FastAPI with PostgreSQL, Redis caching, and Docker Compose files.")